In [ ]:
import os
import re
import csv
import random
import shutil
from datetime import datetime
import torch
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from tqdm import tqdm

from tree_sitter import Language, Parser
import tree_sitter_java

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(2137)

c:\Projects\CS4928-Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


: 

In [ ]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/singleton"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text, batch_size=1):
    inputs = tokenizer(
        text,
        truncation=True,
        max_length=512,
        return_overflowing_tokens=True,
        padding=False
    )

    chunks = inputs["input_ids"]

    print("Number of chunks:", len(chunks))
    print("Maximum chunk length:", max(map(len, chunks)))
    print("Minimum chunk length:", min(map(len, chunks)))
    
    embeddings = []
    weights = []

    for i in range(0, len(chunks), batch_size):
        batch_chunks = chunks[i:i + batch_size]

        batch = tokenizer.pad(
            {"input_ids": batch_chunks},
            padding=True,
            return_tensors="pt"
        )

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        with torch.inference_mode():
            outputs = model(**batch)

            mask = batch["attention_mask"].unsqueeze(-1)

            chunk_embeddings = (
                (outputs.last_hidden_state * mask).sum(dim=1)
                / mask.sum(dim=1)
            )

            chunk_weights = batch["attention_mask"].sum(dim=1)

        embeddings.append(chunk_embeddings.cpu())
        weights.append(chunk_weights.cpu())

    embeddings = torch.cat(embeddings, dim=0)
    weights = torch.cat(weights).to(embeddings.dtype)

    embedding = (
        embeddings * weights.unsqueeze(1)
    ).sum(dim=0) / weights.sum()

    return embedding.numpy()[None, :]

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []

class_folders = {"non-pattern": 0, "pattern": 1}

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    for program_file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                java_programs.append(f.read())

            program_labels.append(label)
            program_files.append(program_file)

JAVA_LANGUAGE = Language(tree_sitter_java.language())
parser = Parser(JAVA_LANGUAGE)

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    source = program.encode("utf-8")
    tree = parser.parse(source)

    scopes = []

    def traverse(node):
        if node.type in {
            "method_declaration",
            "constructor_delcaration"
        }:
            scopes.append({
                "type": node.type,
                "code": source[node.start_byte:node.end_byte].decode("utf-8"),
                "start_line": node.start_point.row + 1,
                "end_line": node.end_point.row + 1
            })
        for child in node.children:
            traverse(child)

    traverse(tree.root_node)

    scope_embeddings = [get_embeddings(scope["code"]) 
                        for scope in scopes
                        ]

    if scope_embeddings:
        program_embed = np.mean(scope_embeddings, axis=0)
    else:
        program_embed = get_embeddings(program)

    program_embeddings.append(program_embed)

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# Get k-nearest neighbors indices for each program
k = 5  # Number of neighbors to consider
knn_indices = np.argsort(distance_matrix)[:, 1:k+1]

# Initialize an array to store predicted labels
predicted_labels = []

# Predict labels for each program based on the majority label of neighbors
for indices in knn_indices:
    neighbor_labels = [program_labels[i] for i in indices]
    predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
    predicted_labels.append(predicted_label)

# Convert predicted_labels to NumPy array for further analysis
predicted_labels = np.array(predicted_labels)

# Split the data into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(normalized_embeddings, predicted_labels, test_size=0.2, random_state=0)

X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
    normalized_embeddings,
    program_labels,
    program_files,
    test_size=0.2,
    random_state=0
)

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
predictions = knn.predict(X_test)

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 963.95it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Generating embeddings:   0%|          | 0/30 [00:00<?, ?it/s]

Number of chunks: 2
Maximum chunk length: 512
Minimum chunk length: 5


Generating embeddings:   3%|▎         | 1/30 [00:00<00:09,  2.96it/s]

Number of chunks: 1
Maximum chunk length: 8
Minimum chunk length: 8
Number of chunks: 1
Maximum chunk length: 12
Minimum chunk length: 12
Number of chunks: 1
Maximum chunk length: 15
Minimum chunk length: 15
Number of chunks: 1
Maximum chunk length: 29
Minimum chunk length: 29
Number of chunks: 1
Maximum chunk length: 28
Minimum chunk length: 28
Number of chunks: 1
Maximum chunk length: 21
Minimum chunk length: 21
Number of chunks: 1
Maximum chunk length: 25
Minimum chunk length: 25
Number of chunks: 1
Maximum chunk length: 11
Minimum chunk length: 11
Number of chunks: 1
Maximum chunk length: 17
Minimum chunk length: 17
Number of chunks: 1
Maximum chunk length: 18
Minimum chunk length: 18
Number of chunks: 1
Maximum chunk length: 13
Minimum chunk length: 13
Number of chunks: 1
Maximum chunk length: 17
Minimum chunk length: 17
Number of chunks: 1
Maximum chunk length: 18
Minimum chunk length: 18
Number of chunks: 1
Maximum chunk length: 18
Minimum chunk length: 18
Number of chunks: 1
Ma

Generating embeddings:  10%|█         | 3/30 [00:00<00:07,  3.57it/s]

...

Generating embeddings:  13%|█▎        | 4/30 [00:01<00:06,  4.10it/s]

Number of chunks: 1
Maximum chunk length: 31
Minimum chunk length: 31
Number of chunks: 1
Maximum chunk length: 38
Minimum chunk length: 38
Number of chunks: 1
Maximum chunk length: 35
Minimum chunk length: 35
Number of chunks: 1
Maximum chunk length: 35
Minimum chunk length: 35
Number of chunks: 1
Maximum chunk length: 38
Minimum chunk length: 38


Generating embeddings:  17%|█▋        | 5/30 [00:01<00:06,  3.66it/s]

Number of chunks: 1
Maximum chunk length: 35
Minimum chunk length: 35
Number of chunks: 1
Maximum chunk length: 30
Minimum chunk length: 30
Number of chunks: 1
Maximum chunk length: 30
Minimum chunk length: 30
Number of chunks: 1
Maximum chunk length: 57
Minimum chunk length: 57
Number of chunks: 1
Maximum chunk length: 430
Minimum chunk length: 430
Number of chunks: 1
Maximum chunk length: 96
Minimum chunk length: 96
Number of chunks: 2
Maximum chunk length: 512
Minimum chunk length: 4
Number of chunks: 1
Maximum chunk length: 362
Minimum chunk length: 362


Generating embeddings:  20%|██        | 6/30 [00:02<00:11,  2.13it/s]